In [ ]:
# RUNTIME: CPU (standard) - do NOT pick a GPU
# POST-HOC, descriptive: what the relocation / method-change exclusion rule caught. No model is fitted.
import os
import pandas as pd
ROOT = os.environ.get("PAPER_ROOT", ".")
IN = os.path.join(ROOT, "data", "derived")
OUT = os.path.join(ROOT, "outputs")
os.makedirs(OUT, exist_ok=True)
man = pd.read_parquet(os.path.join(IN, "manifest.parquet"))
sp = pd.read_parquet(os.path.join(IN, "sampling_points.parquet"))
x = man[man["excluded_reason"] == "station moved or method changed"]
print(len(man), "series in the manifest;", len(x), "excluded by the relocation / method-change rule")

In [ ]:
# E1: counts by reason, and whether a break year and a meteorological match exist
summ = x.groupby("pollutant").apply(lambda g: pd.Series({
    "n_excluded": len(g), "moved_only": int((g["moved"] & ~g["method_change"]).sum()),
    "method_change_only": int((~g["moved"] & g["method_change"]).sum()), "both": int((g["moved"] & g["method_change"]).sum()),
    "with_break_year": int(g["break_year"].notna().sum()), "with_met_match": int(g["isd"].notna().sum())})).reset_index()
summ.to_csv(os.path.join(OUT, "posthoc_exclusions_summary.csv"), index=False)
print(summ.to_string(index=False))

In [ ]:
# E2: the sequence of measurement-method codes in the EEA metadata, for series excluded by a method change
s = sp[sp["sp"].isin(x.loc[x["method_change"], "sp"])].sort_values("Year")
seq = (s.groupby(["pollutant", "sp"])["mmethod"].apply(lambda v: " -> ".join(dict.fromkeys(v.dropna().astype(str))))
         .rename("method_sequence").reset_index())
tab = seq.groupby(["pollutant", "method_sequence"]).size().rename("n").reset_index()
tab["share_of_pollutant"] = tab["n"] / tab.groupby("pollutant")["n"].transform("sum")
tab = tab.sort_values(["pollutant", "n"], ascending=[True, False])
tab.to_csv(os.path.join(OUT, "posthoc_exclusions_methods.csv"), index=False)
print(tab.groupby("pollutant").head(8).to_string(index=False))

In [ ]:
# MANIFEST rows
nb = "10_exclusion_audit_CPU"
rows_md = ["| file | notebook | cell | description |", "|---|---|---|---|",
    f"| posthoc_exclusions_summary.csv | {nb} | 1 | POST-HOC, descriptive: series excluded by the relocation / method-change rule, by reason, with break year and meteorological match |",
    f"| posthoc_exclusions_methods.csv | {nb} | 2 | POST-HOC, descriptive: sequences of EEA measurement-method codes among series excluded for a method change |"]
open(os.path.join(OUT, "MANIFEST.md"), "w").write("\n".join(rows_md) + "\n")
print("manifest written")